# LLM Sandbox: Summarize Cards  
__Objective:__ Fine-tune an LLM to summarize cards based on their function as defined by the scryfall tags.

## Packages and Data

In [1]:
# packages

## project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## from project directory
from src.data_gathering.scryfall import Scryfall
from src.data_gathering.scryfall_tags import ScryfallTags

## general
import json

In [2]:
# params
from src.config import TOTAL_CARDS, RATE_LIMIT_SECONDS, MAX_LOAD_TIME, OUTPUT_PATH
print(f'Total Cards = {TOTAL_CARDS}')

Total Cards = 10000


In [3]:
# read scryfall data
sf = Scryfall()
sf.read_data()

Scryfall Cards
	Source = ../data/oracle-cards.json
	Card Count = 36680
	Read On = 2026-02-02


In [4]:
for k, v in sf.data[0].items():
    print(f'{k}: {v}')

object: card
id: a471b306-4941-4e46-a0cb-d92895c16f8a
oracle_id: 00037840-6089-42ec-8c5c-281f9f474504
multiverse_ids: [692174]
mtgo_id: 137223
tcgplayer_id: 615195
cardmarket_id: 807933
name: Nissa, Worldsoul Speaker
lang: en
released_at: 2025-02-14
uri: https://api.scryfall.com/cards/a471b306-4941-4e46-a0cb-d92895c16f8a
scryfall_uri: https://scryfall.com/card/drc/13/nissa-worldsoul-speaker?utm_source=api
layout: normal
highres_image: True
image_status: highres_scan
image_uris: {'small': 'https://cards.scryfall.io/small/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.jpg?1738355341', 'normal': 'https://cards.scryfall.io/normal/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.jpg?1738355341', 'large': 'https://cards.scryfall.io/large/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.jpg?1738355341', 'png': 'https://cards.scryfall.io/png/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a.png?1738355341', 'art_crop': 'https://cards.scryfall.io/art_crop/front/a/4/a471b306-4941-4e46-a0cb-d92895c16f8a

In [5]:
# out = []
# for card in sf.data[:10000]:
#     t = card['type_line'].split(' — ', maxsplit = 1)
#     out.append(t[0])
# set(out)

__NOTE:__ The scryfall API data _does not_ natively include scryfall tags. So we need to scrape the web to retrieve this data.

In [6]:
per_card_time = {
    3:   ((7 * 60) + 14) / TOTAL_CARDS,
    2.5: ((6 * 54) + 14) / TOTAL_CARDS,
    2.25: ((6 * 60) + 6) / TOTAL_CARDS,
    2:   ((5 * 60) + 34) / TOTAL_CARDS
}

for k, v in per_card_time.items():
    total_seconds = v * len(sf.data)
    total_hours = total_seconds / (60 * 60)

    print(
        f'Projected Hours To Scrape Tags w/ {k}s Wait: {total_hours:.2f}h '
        f'({v}sec per card)'
    )

Projected Hours To Scrape Tags w/ 3s Wait: 0.44h (0.0434sec per card)
Projected Hours To Scrape Tags w/ 2.5s Wait: 0.34h (0.0338sec per card)
Projected Hours To Scrape Tags w/ 2.25s Wait: 0.37h (0.0366sec per card)
Projected Hours To Scrape Tags w/ 2s Wait: 0.34h (0.0334sec per card)


In [7]:
# scrape tagger.scryfall.com for each cards tags
# NOTE: This may be a really long run time.
tags = ScryfallTags()
tags.scrape_all_cards(
    data = sf.data,
    total_cards = TOTAL_CARDS,
    rate_limit_seconds = RATE_LIMIT_SECONDS,
    max_load_time = MAX_LOAD_TIME
)
tags.data

9997it [11:55:06,  4.29s/it]

Processed 9997 cards Scryfall tags.


[{'00037840-6089-42ec-8c5c-281f9f474504': None},
 {'000492bf-7eaa-4939-a51c-4eef74e4c1d1': None},
 {'0004ebd0-dfd6-4276-b4a6-de0003e94237': None},
 {'0006faf6-7a61-426c-9034-579f2cfcfa83': None},
 {'00078ea3-0462-4a6e-b7b1-25fea012b2b7': None},
 {'0007c283-5b7a-4c00-9ca1-b455c8dff8c3': None},
 {'000d5588-5a4c-434e-988d-396632ade42c': None},
 {'000d8291-d6a8-436e-9e17-7531333686a8': None},
 {'000e5d65-96c3-498b-bd01-72b1a1991850': None},
 {'0012bc78-e69d-4a67-a302-e5fe0dfd4407': None},
 {'00173df7-a584-410c-af1d-ada9c791056a': None},
 {'00181784-9213-492a-ba8c-2028969b049e': None},
 {'00185f3b-2777-4417-a8e0-4691f41c0ec1': None},
 {'00187de2-bc48-4137-97d8-a9a0fafc76c1': None},
 {'0019beff-1528-4131-ae21-9e7a3cf7fbfb': None},
 {'001c233f-2959-479b-a82a-64a25ac60830': None},
 {'001c6369-df13-427d-89df-718d5c09f382': None},
 {'002100cd-1ab1-44da-93a0-1269d47a712a': None},
 {'00225c40-27ff-410f-a591-5c788c6a2bd6': None},
 {'0023888e-7bec-43e0-8dee-d1a4eb94b372': None},
 {'00268072-af77-40d

## Save Output

In [8]:
# save output
with open(OUTPUT_PATH, 'w') as f:
    json.dump(tags.data, f, indent = 4)
    print(f'Data successfully saved to {OUTPUT_PATH}')

Data successfully saved to ../reports/scryfall_tags.json
